In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics -q

In [ ]:
import zipfile, os

zip_path = '/content/drive/MyDrive/gesture-dataset.zip'
extract_path = '/content/gesture'

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)

print('Extracted files:')
for root, dirs, files in os.walk(extract_path):
    level = root.replace(extract_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')

In [ ]:
import os

dataset_path = '/content/gesture/dataset'
for split in ['train', 'val', 'test']:
    imgs = len(os.listdir(f'{dataset_path}/images/{split}'))
    lbls = len(os.listdir(f'{dataset_path}/labels/{split}'))
    print(f'{split}: {imgs} images, {lbls} labels')

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/gesture/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device='0',
    project='/content/runs',
    name='gesture_v1',
    patience=15,
    save=True,
    plots=True,
)

print(f"Best mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")

In [ ]:
best_model = YOLO('/content/runs/gesture_v1/weights/best.pt')
metrics = best_model.val(data='/content/gesture/data.yaml', split='test')
print(f'Test mAP50: {metrics.box.map50:.4f}')

In [ ]:
best_model.export(
    format='onnx',
    dynamic=True,
    simplify=True,
    imgsz=640,
)
print('ONNX export complete: runs/gesture_v1/weights/best.onnx')

In [ ]:
import shutil

src = '/content/runs/gesture_v1/weights/best.onnx'
dst = '/content/drive/MyDrive/gesture.onnx'
shutil.copy(src, dst)
print(f'Saved to Google Drive: {dst}')